# 3.47 — Imbalanced Data

Imbalanced data appears when one class is common and another class is rare, so ordinary empirical risk can look excellent while the rare class receives almost no learning signal. In this lesson, we rebuild the practical fixes — class weights, resampling, SMOTE-style synthetic points, and threshold selection — from NumPy arrays so the math stays visible: every method changes either which examples the model sees, how strongly each example counts, or where a final probability becomes a decision.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build imbalanced-data handling one idea at a time. Run each cell in order and inspect the printed counts, weights, and losses — the main habit is to ask which examples are contributing force to the objective. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, and vectorized loss arithmetic.
import matplotlib.pyplot as plt  # compact visual checks for distributions and decision thresholds.
np.random.seed(0)  # reproducibility for synthetic samples and resampling.

### 1. Imbalance changes what the average loss asks for

Empirical risk averages per-example losses. If 95 negatives and 5 positives all count equally, then the majority class owns 95% of the objective. A classifier that ignores positives can therefore look deceptively good by accuracy, even though its rare-class recall is zero. We first make that arithmetic explicit.

In [ ]:
n0_w, n1_w = 95, 5  # 95 majority examples, 5 minority examples.
y_w = np.r_[np.zeros(n0_w, dtype=int), np.ones(n1_w, dtype=int)]  # binary labels.
predict_all_zero_w = np.zeros_like(y_w)  # a naive model that always predicts the majority class.
print("class counts:", np.bincount(y_w))  # inspect imbalance.
print("positive fraction:", round(float(y_w.mean()), 3))  # base rate of the rare class.

▶ What you'll see: the positive class is only 5% of the data, so it is easy for an average to hide.

In [ ]:
accuracy_w = np.mean(predict_all_zero_w == y_w)  # majority-class accuracy.
recall_pos_w = np.sum((predict_all_zero_w == 1) & (y_w == 1)) / np.sum(y_w == 1)  # rare-class recall.
print("accuracy:", round(float(accuracy_w), 3))
print("positive recall:", round(float(recall_pos_w), 3))
assert round(float(accuracy_w), 3) == 0.950
assert round(float(recall_pos_w), 3) == 0.000

▶ What you'll see: 95% accuracy coexists with 0% recall on the class we probably care about.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["majority 0", "minority 1"], np.bincount(y_w), color=["steelblue", "crimson"])
plt.title("1: class counts drive the unweighted average")
plt.ylabel("examples")
plt.show()

▶ What you'll see: the majority bar visually dominates the training objective.

*Why it's done this way:* ERM minimizes an average, and an average is a vote weighted by frequency. When the rare class has only 5 votes out of 100, the unweighted objective can improve mostly by serving the majority; imbalance handling is therefore not cosmetic, it is a deliberate rewrite of what the average is allowed to value.

### 2. Class weights change the loss contribution, not the labels

The weighted loss in the lesson is

$$L=\frac{1}{m}\sum_i w_{y_i}\ell(f(x_i),y_i).$$

The label stays the same, the model stays the same, but the multiplier $w_{y_i}$ changes how loudly each example speaks. A common balanced choice sets each class's total weight roughly equal: $w_c=m/(K n_c)$ for $K$ classes.

In [ ]:
counts_w = np.bincount(y_w)  # [95, 5].
K_w = len(counts_w)  # two classes.
m_w = len(y_w)  # total examples.
class_weights_w = m_w / (K_w * counts_w)  # balanced class weights.
print("class weights:", np.round(class_weights_w, 3))
assert np.allclose(np.round(class_weights_w, 3), [0.526, 10.000])

▶ What you'll see: each minority example gets weight 10, while each majority example gets about 0.526.

In [ ]:
example_weights_w = class_weights_w[y_w]  # attach each label's class weight to each row.
print("total majority weight:", round(float(example_weights_w[y_w == 0].sum()), 3))
print("total minority weight:", round(float(example_weights_w[y_w == 1].sum()), 3))
assert round(float(example_weights_w[y_w == 0].sum()), 3) == 50.000
assert round(float(example_weights_w[y_w == 1].sum()), 3) == 50.000

▶ What you'll see: 95 majority rows and 5 minority rows now contribute equal total weight.

In [ ]:
losses_w = np.where(y_w == 1, 0.80, 0.20)  # toy per-example losses: positives are currently harder.
unweighted_w = float(np.mean(losses_w))  # ordinary empirical risk.
weighted_w = float(np.mean(example_weights_w * losses_w))  # lesson formula.
print("unweighted risk:", round(unweighted_w, 3))
print("weighted risk:", round(weighted_w, 3))
assert round(unweighted_w, 3) == 0.230
assert round(weighted_w, 3) == 0.500

▶ What you'll see: the same per-example losses look much worse once the rare-class mistakes are allowed to matter.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["unweighted", "weighted"], [unweighted_w, weighted_w], color=["gray", "seagreen"])
plt.title("2: class weights rewrite the objective")
plt.ylabel("average loss")
plt.show()

▶ What you'll see: the weighted objective rises because it no longer lets many easy negatives swamp few hard positives.

*Why it's done this way:* The formula multiplies loss before averaging, so it changes the gradient force without duplicating data. Balanced weights make each class's total contribution comparable, which is exactly the mathematical antidote to frequency dominance.

### 3. Resampling changes the training distribution directly

Another way to change the objective is to change the rows presented to the learner. Oversampling repeats minority examples until class counts match; undersampling discards majority examples until counts match. The risk being optimized is then the ordinary average over a different empirical distribution.

In [ ]:
rng_w = np.random.default_rng(0)
idx0_w = np.where(y_w == 0)[0]  # majority indices.
idx1_w = np.where(y_w == 1)[0]  # minority indices.
minority_extra_w = rng_w.choice(idx1_w, size=n0_w - n1_w, replace=True)  # repeat positives.
oversampled_idx_w = np.r_[np.arange(m_w), minority_extra_w]
y_over_w = y_w[oversampled_idx_w]
print("oversampled counts:", np.bincount(y_over_w))
assert np.all(np.bincount(y_over_w) == [95, 95])

▶ What you'll see: random oversampling creates a balanced 190-row training set by repeating positives.

In [ ]:
majority_keep_w = rng_w.choice(idx0_w, size=n1_w, replace=False)  # keep only 5 negatives.
undersampled_idx_w = np.r_[majority_keep_w, idx1_w]
y_under_w = y_w[undersampled_idx_w]
print("undersampled counts:", np.bincount(y_under_w))
assert np.all(np.bincount(y_under_w) == [5, 5])

▶ What you'll see: random undersampling creates a balanced 10-row training set by throwing away most negatives.

In [ ]:
plt.figure(figsize=(5, 3))
width_w = 0.25
x_w = np.arange(2)
plt.bar(x_w - width_w, np.bincount(y_w), width_w, label="original")
plt.bar(x_w, np.bincount(y_over_w), width_w, label="over")
plt.bar(x_w + width_w, np.bincount(y_under_w), width_w, label="under")
plt.xticks(x_w, ["class 0", "class 1"])
plt.title("3: resampling changes class counts")
plt.ylabel("training rows")
plt.legend()
plt.show()

▶ What you'll see: oversampling grows the rare class; undersampling shrinks the common class.

*Why it's done this way:* Resampling is equivalent to changing the empirical distribution before the average is computed. Oversampling preserves all majority evidence but can duplicate minority noise; undersampling avoids duplicates but increases variance because it discards many majority rows.

### 4. SMOTE synthesizes minority points between neighbors

Random oversampling repeats exact rows. SMOTE instead creates a new minority point on the line segment between a minority example and a minority neighbor:

$$x_{new}=x_i+\lambda(x_j-x_i),\qquad 0\le \lambda\le 1.$$

The point is still in the minority region, but it is not a duplicate, so the learner sees a smoother minority cloud.

In [ ]:
X_min_w = np.array([[2.0, 2.0], [2.4, 2.1], [2.2, 2.6], [2.8, 2.7], [3.0, 2.3]])
base_w = X_min_w[0]
neighbor_w = X_min_w[2]
lam_w = 0.35
synthetic_w = base_w + lam_w * (neighbor_w - base_w)
print("base:", base_w, "neighbor:", neighbor_w)
print("synthetic:", np.round(synthetic_w, 3))
assert np.allclose(np.round(synthetic_w, 3), [2.070, 2.210])

▶ What you'll see: the synthetic point lies 35% of the way from `[2.0, 2.0]` toward `[2.2, 2.6]`.

In [ ]:
rng_smote_w = np.random.default_rng(2)
synthetics_w = []
for t_w in range(20):
    i_w = rng_smote_w.integers(0, len(X_min_w))
    j_w = (i_w + rng_smote_w.integers(1, len(X_min_w))) % len(X_min_w)
    lam_t_w = rng_smote_w.random()
    synthetics_w.append(X_min_w[i_w] + lam_t_w * (X_min_w[j_w] - X_min_w[i_w]))
synthetics_w = np.array(synthetics_w)
print("synthetic shape:", synthetics_w.shape)
print("first synthetic:", np.round(synthetics_w[0], 3))

▶ What you'll see: several new minority points, each built by interpolation rather than copying.

In [ ]:
plt.figure(figsize=(4.6, 3.6))
plt.scatter(X_min_w[:, 0], X_min_w[:, 1], s=80, label="minority originals", color="crimson")
plt.scatter(synthetics_w[:, 0], synthetics_w[:, 1], s=35, label="SMOTE synthetics", color="orange")
plt.plot([base_w[0], neighbor_w[0]], [base_w[1], neighbor_w[1]], color="black", linestyle="--", linewidth=1)
plt.title("4: SMOTE points live between minority neighbors")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.legend()
plt.show()

▶ What you'll see: orange points fill the minority region instead of stacking directly on red originals.

*Why it's done this way:* Linear interpolation assumes nearby minority examples share a local class region. The formula creates new support inside that region, which can smooth a decision boundary, but it can also be dangerous if the minority neighbors straddle majority territory.

### 5. Thresholds and validation decide whether the fix worked

Imbalance methods change training, but the deployment decision still depends on a threshold. Lowering the threshold usually increases recall and decreases precision. The validation set is where we choose the tradeoff, because a lower weighted loss is not useful if the final decisions fail the rare-class objective.

In [ ]:
scores_w = np.array([0.05, 0.12, 0.18, 0.22, 0.31, 0.40, 0.55, 0.63, 0.72, 0.90])
y_val_w = np.array([0, 0, 0, 1, 0, 1, 0, 1, 1, 1])
thresholds_w = np.array([0.20, 0.50, 0.70])
print("validation positives:", int(y_val_w.sum()))
print("thresholds:", thresholds_w)

▶ What you'll see: five positives are scattered across a probability-like score scale.

In [ ]:
precisions_w, recalls_w, f1s_w = [], [], []
for th_w in thresholds_w:
    pred_w = scores_w >= th_w
    tp_w = np.sum(pred_w & (y_val_w == 1))
    fp_w = np.sum(pred_w & (y_val_w == 0))
    fn_w = np.sum((~pred_w) & (y_val_w == 1))
    precision_w = tp_w / (tp_w + fp_w) if tp_w + fp_w else 0.0
    recall_w = tp_w / (tp_w + fn_w) if tp_w + fn_w else 0.0
    f1_w = 2 * precision_w * recall_w / (precision_w + recall_w) if precision_w + recall_w else 0.0
    precisions_w.append(precision_w); recalls_w.append(recall_w); f1s_w.append(f1_w)
print("precision:", np.round(precisions_w, 3))
print("recall:", np.round(recalls_w, 3))
print("F1:", np.round(f1s_w, 3))
assert np.allclose(np.round(f1s_w, 3), [0.833, 0.667, 0.571])

▶ What you'll see: threshold 0.20 wins F1 here because it recovers all positives while adding only two false alarms.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(thresholds_w, precisions_w, marker="o", label="precision")
plt.plot(thresholds_w, recalls_w, marker="s", label="recall")
plt.plot(thresholds_w, f1s_w, marker="^", label="F1")
plt.title("5: threshold choice changes rare-class behavior")
plt.xlabel("decision threshold")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

▶ What you'll see: recall falls as the threshold rises, while precision and F1 move according to the validation tradeoff.

*Why it's done this way:* Training weights and resampling shape the score function, but the threshold shapes the actual action. Validation keeps us honest by measuring the cost-sensitive decision we care about on data not used to create the balance trick.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small arrays, prints intermediate values with inline `# ->` checks, draws one picture, and includes an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Accuracy can hide rare-class failure

When negatives dominate, predicting the majority class can score well on accuracy while completely missing positives. Count both metrics on ten labels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
print("true labels:", t1_y.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
t1_counts = np.bincount(t1_y)
print("class counts:", t1_counts.tolist())  # -> [8, 2]
t1_pred = np.zeros_like(t1_y)
print("all-zero predictions:", t1_pred.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
t1_correct = t1_pred == t1_y
print("correct mask:", t1_correct.astype(int).tolist())  # -> [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
t1_accuracy = float(np.mean(t1_correct))
print("accuracy:", round(t1_accuracy, 3))  # -> 0.8
t1_true_positives = int(np.sum((t1_pred == 1) & (t1_y == 1)))
print("true positives:", t1_true_positives)  # -> 0
t1_actual_positives = int(np.sum(t1_y == 1))
print("actual positives:", t1_actual_positives)  # -> 2
t1_recall = t1_true_positives / t1_actual_positives
print("positive recall:", round(t1_recall, 3))  # -> 0.0

plt.figure(figsize=(4.4, 3))
plt.bar(["class 0", "class 1"], t1_counts, color=["steelblue", "crimson"])
plt.ylabel("examples")
plt.title("Toy 1 · majority dominates counts")
plt.show()
assert round(t1_accuracy, 3) == 0.8 and t1_recall == 0.0

▶ What you'll see: `80%` accuracy still has `0.0` recall for the rare class.

### ✍️ Toy 2 · Class weights equalize total class influence

Balanced class weights make the total weight of each class comparable before losses are averaged. The same per-example losses then produce a much larger objective.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
print("labels:", t2_y.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
t2_counts = np.bincount(t2_y)
print("class counts:", t2_counts.tolist())  # -> [8, 2]
t2_K = len(t2_counts)
print("number of classes:", t2_K)  # -> 2
t2_m = len(t2_y)
print("number of examples:", t2_m)  # -> 10
t2_class_weights = t2_m / (t2_K * t2_counts)
print("class weights:", t2_class_weights.tolist())  # -> [0.625, 2.5]
t2_example_weights = t2_class_weights[t2_y]
print("example weights:", t2_example_weights.tolist())  # -> [0.625, 0.625, 0.625, 0.625, 0.625, 0.625, 0.625, 0.625, 2.5, 2.5]
t2_loss = np.where(t2_y == 1, 0.9, 0.1)
print("per-example losses:", t2_loss.tolist())  # -> [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.9, 0.9]
t2_majority_weight = float(t2_example_weights[t2_y == 0].sum())
print("total majority weight:", round(t2_majority_weight, 3))  # -> 5.0
t2_minority_weight = float(t2_example_weights[t2_y == 1].sum())
print("total minority weight:", round(t2_minority_weight, 3))  # -> 5.0
t2_unweighted = float(t2_loss.mean())
print("unweighted risk:", round(t2_unweighted, 3))  # -> 0.26
t2_weighted = float(np.mean(t2_example_weights * t2_loss))
print("weighted risk:", round(t2_weighted, 3))  # -> 0.5

plt.figure(figsize=(4.4, 3))
plt.bar(["unweighted", "weighted"], [t2_unweighted, t2_weighted], color=["gray", "seagreen"])
plt.ylabel("average loss")
plt.title("Toy 2 · weights change the objective")
plt.show()
assert round(t2_weighted, 3) == 0.5 and round(t2_majority_weight, 3) == round(t2_minority_weight, 3)

▶ What you'll see: total class weights are both `5.0`, so hard positive losses now matter as much as many easy negatives.

### ✍️ Toy 3 · Resampling changes the row distribution

Oversampling repeats rare rows until counts match. Undersampling keeps only a small random slice of the majority class.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
print("original labels:", t3_y.tolist())  # -> [0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
t3_idx0 = np.where(t3_y == 0)[0]
print("majority indices:", t3_idx0.tolist())  # -> [0, 1, 2, 3, 4, 5, 6, 7]
t3_idx1 = np.where(t3_y == 1)[0]
print("minority indices:", t3_idx1.tolist())  # -> [8, 9]
t3_extra = t3_rng.choice(t3_idx1, size=len(t3_idx0) - len(t3_idx1), replace=True)
print("oversampled extra indices:", t3_extra.tolist())  # -> [9, 9, 9, 8, 8, 8]
t3_over_idx = np.r_[np.arange(len(t3_y)), t3_extra]
print("oversampled row count:", int(t3_over_idx.size))  # -> 16
t3_y_over = t3_y[t3_over_idx]
print("oversampled counts:", np.bincount(t3_y_over).tolist())  # -> [8, 8]
t3_keep = t3_rng.choice(t3_idx0, size=len(t3_idx1), replace=False)
print("undersampled majority keep:", t3_keep.tolist())  # -> [7, 0]
t3_under_idx = np.r_[t3_keep, t3_idx1]
print("undersampled row count:", int(t3_under_idx.size))  # -> 4
t3_y_under = t3_y[t3_under_idx]
print("undersampled counts:", np.bincount(t3_y_under).tolist())  # -> [2, 2]

plt.figure(figsize=(5, 3))
t3_x = np.arange(2)
plt.bar(t3_x - 0.22, np.bincount(t3_y), width=0.22, label="original")
plt.bar(t3_x, np.bincount(t3_y_over), width=0.22, label="over")
plt.bar(t3_x + 0.22, np.bincount(t3_y_under), width=0.22, label="under")
plt.xticks(t3_x, ["class 0", "class 1"])
plt.ylabel("rows")
plt.title("Toy 3 · resampling balances counts")
plt.legend()
plt.show()
assert np.array_equal(np.bincount(t3_y_over), [8, 8]) and np.array_equal(np.bincount(t3_y_under), [2, 2])

▶ What you'll see: oversampling grows the minority class to `8`, while undersampling shrinks the majority class to `2`.

### ✍️ Toy 4 · SMOTE interpolates between minority neighbors

SMOTE does not copy a minority point. It chooses two minority neighbors and creates a new point along the segment between them.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_X_min = np.array([[2., 1.], [3., 1.], [2., 3.], [4., 3.], [3., 4.], [5., 4.]])
print("minority points:", t4_X_min.tolist())  # -> [[2.0, 1.0], [3.0, 1.0], [2.0, 3.0], [4.0, 3.0], [3.0, 4.0], [5.0, 4.0]]
t4_base = t4_X_min[0]
print("base point:", t4_base.tolist())  # -> [2.0, 1.0]
t4_neighbor = t4_X_min[2]
print("neighbor point:", t4_neighbor.tolist())  # -> [2.0, 3.0]
t4_lambda = 0.25
print("lambda:", t4_lambda)  # -> 0.25
t4_synthetic = t4_base + t4_lambda * (t4_neighbor - t4_base)
print("one synthetic point:", t4_synthetic.tolist())  # -> [2.0, 1.5]
t4_synthetic_batch = []
for t4_step in range(6):
    t4_i = t4_rng.integers(0, len(t4_X_min))
    t4_j = (t4_i + t4_rng.integers(1, len(t4_X_min))) % len(t4_X_min)
    t4_mix = t4_rng.random()
    t4_point = t4_X_min[t4_i] + t4_mix * (t4_X_min[t4_j] - t4_X_min[t4_i])
    t4_synthetic_batch.append(t4_point)
t4_synthetic_batch = np.array(t4_synthetic_batch)
print("synthetic batch shape:", t4_synthetic_batch.shape)  # -> (6, 2)
print("first batch point:", np.round(t4_synthetic_batch[0], 3).tolist())  # -> [4.73, 3.73]

plt.figure(figsize=(4.8, 3.6))
plt.scatter(t4_X_min[:, 0], t4_X_min[:, 1], s=80, color="crimson", label="minority originals")
plt.scatter(t4_synthetic_batch[:, 0], t4_synthetic_batch[:, 1], s=45, color="orange", label="SMOTE points")
plt.plot([t4_base[0], t4_neighbor[0]], [t4_base[1], t4_neighbor[1]], color="black", linestyle="--")
plt.title("Toy 4 · SMOTE fills segments")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.legend()
plt.show()
assert np.allclose(t4_synthetic, [2.0, 1.5]) and t4_synthetic_batch.shape == (6, 2)

▶ What you'll see: orange synthetic points appear between red minority originals instead of stacking on top of them.

### ✍️ Toy 5 · Thresholds trade precision for recall

Changing the score cutoff changes which rare positives are recovered and which negatives become false alarms. Validation F1 summarizes that tradeoff.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_scores = np.array([0.10, 0.18, 0.25, 0.35, 0.42, 0.55, 0.68, 0.72, 0.85, 0.92])
print("validation scores:", t5_scores.tolist())  # -> [0.1, 0.18, 0.25, 0.35, 0.42, 0.55, 0.68, 0.72, 0.85, 0.92]
t5_y = np.array([0, 0, 1, 0, 1, 0, 1, 1, 1, 1])
print("validation labels:", t5_y.tolist())  # -> [0, 0, 1, 0, 1, 0, 1, 1, 1, 1]
t5_thresholds = np.array([0.30, 0.50, 0.70])
print("thresholds:", t5_thresholds.tolist())  # -> [0.3, 0.5, 0.7]
t5_precisions = []
t5_recalls = []
t5_f1s = []
t5_counts = []
for t5_threshold in t5_thresholds:
    t5_pred = t5_scores >= t5_threshold
    t5_tp = int(np.sum(t5_pred & (t5_y == 1)))
    t5_fp = int(np.sum(t5_pred & (t5_y == 0)))
    t5_fn = int(np.sum((~t5_pred) & (t5_y == 1)))
    t5_precision = t5_tp / (t5_tp + t5_fp) if (t5_tp + t5_fp) else 0.0
    t5_recall = t5_tp / (t5_tp + t5_fn) if (t5_tp + t5_fn) else 0.0
    t5_f1 = 2 * t5_precision * t5_recall / (t5_precision + t5_recall) if (t5_precision + t5_recall) else 0.0
    t5_precisions.append(t5_precision)
    t5_recalls.append(t5_recall)
    t5_f1s.append(t5_f1)
    t5_counts.append((t5_tp, t5_fp, t5_fn))
print("tp/fp/fn counts:", t5_counts)  # -> [(5, 2, 1), (4, 1, 2), (3, 0, 3)]
print("precision:", np.round(t5_precisions, 3).tolist())  # -> [0.714, 0.8, 1.0]
print("recall:", np.round(t5_recalls, 3).tolist())  # -> [0.833, 0.667, 0.5]
print("F1:", np.round(t5_f1s, 3).tolist())  # -> [0.769, 0.727, 0.667]
t5_best_threshold = float(t5_thresholds[int(np.argmax(t5_f1s))])
print("best threshold:", t5_best_threshold)  # -> 0.3

plt.figure(figsize=(5, 3))
plt.plot(t5_thresholds, t5_precisions, marker="o", label="precision")
plt.plot(t5_thresholds, t5_recalls, marker="s", label="recall")
plt.plot(t5_thresholds, t5_f1s, marker="^", label="F1")
plt.xlabel("threshold")
plt.ylim(0, 1.05)
plt.title("Toy 5 · validation threshold sweep")
plt.legend()
plt.show()
assert np.allclose(np.round(t5_f1s, 3), [0.769, 0.727, 0.667]) and t5_best_threshold == 0.3

▶ What you'll see: the lowest threshold has the best F1 here because it recovers more positives with only two false alarms.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, masks, resampling, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for class-count, loss, and threshold visualizations.
np.random.seed(0) # make random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Count the classes

**Goal.** Build a tiny imbalanced label vector and count each class, because imbalance is first a frequency problem. We build it in 2 steps.

In [ ]:
y_b1 = np.array([0] * 18 + [1] * 2) # create 20 labels with a 90/10 split.
counts_b1 = np.bincount(y_b1) # count how many examples belong to each class.
print("counts:", counts_b1) # inspect majority and minority counts.
assert np.all(counts_b1 == np.array([18, 2])) # verify the intended imbalance.

▶ What you'll see: class 0 has 18 examples while class 1 has only 2.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact count plot.
plt.bar(["class 0", "class 1"], counts_b1, color=["steelblue", "crimson"]) # show the two class frequencies.
plt.title("Basic 1: class counts") # label the plot.
plt.ylabel("examples") # label the count axis.
plt.show() # display the chart.

▶ What you'll see: the majority bar is nine times taller than the minority bar.

👀 Takeaway: before changing a model, measure the label distribution it is optimizing over.

### Basic 2 — Compute the minority fraction

**Goal.** Convert counts into proportions, because the class prior tells us how much of the unweighted objective belongs to each class. We build it in 2 steps.

In [ ]:
y_b2 = np.array([0] * 18 + [1] * 2) # recreate the 90/10 labels locally.
counts_b2 = np.bincount(y_b2) # count class membership.
fractions_b2 = counts_b2 / len(y_b2) # convert counts into fractions of the dataset.
print("fractions:", np.round(fractions_b2, 3)) # inspect the empirical class prior.
assert np.allclose(fractions_b2, [0.9, 0.1]) # verify the proportions.

▶ What you'll see: the minority class owns only 10% of the examples.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact pie-style bar chart.
plt.bar(["majority share", "minority share"], fractions_b2, color=["gray", "orange"]) # plot objective shares.
plt.ylim(0, 1) # keep the fraction scale visible.
plt.title("Basic 2: class prior") # title the figure.
plt.ylabel("fraction") # label the fraction axis.
plt.show() # display the chart.

▶ What you'll see: an unweighted average spends most of its mass on the majority class.

👀 Takeaway: class proportions are also loss proportions when every example has equal weight.

### Basic 3 — Expose majority-class accuracy

**Goal.** Evaluate a classifier that always predicts the majority class, because high accuracy can hide rare-class failure. We build it in 2 steps.

In [ ]:
y_b3 = np.array([0] * 18 + [1] * 2) # define an imbalanced validation set.
pred_b3 = np.zeros_like(y_b3) # predict the majority class for every example.
accuracy_b3 = np.mean(pred_b3 == y_b3) # compute ordinary accuracy.
print("accuracy:", round(float(accuracy_b3), 3)) # inspect the misleading score.
assert round(float(accuracy_b3), 3) == 0.9 # verify 18/20 correct.

▶ What you'll see: the trivial all-zero classifier gets 90% accuracy.

In [ ]:
minority_recall_b3 = np.sum((pred_b3 == 1) & (y_b3 == 1)) / np.sum(y_b3 == 1) # compute recall for class 1.
print("minority recall:", round(float(minority_recall_b3), 3)) # inspect rare-class performance.
plt.figure(figsize=(4, 3)) # create a score comparison chart.
plt.bar(["accuracy", "minority recall"], [accuracy_b3, minority_recall_b3], color=["gray", "crimson"]) # compare metrics.
plt.ylim(0, 1) # keep both scores on the same scale.
plt.title("Basic 3: accuracy can mislead") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the accuracy bar is high while minority recall is zero.

👀 Takeaway: imbalanced problems need metrics that inspect the rare class directly.

### Basic 4 — Attach balanced class weights

**Goal.** Compute $w_c=m/(K n_c)$, because this makes each class contribute equal total weight to the loss. We build it in 2 steps.

In [ ]:
y_b4 = np.array([0] * 18 + [1] * 2) # recreate the imbalanced labels.
counts_b4 = np.bincount(y_b4) # count examples per class.
weights_b4 = len(y_b4) / (len(counts_b4) * counts_b4) # balanced class-weight formula.
print("class weights:", np.round(weights_b4, 3)) # inspect majority and minority weights.
assert np.allclose(weights_b4, [0.55555556, 5.0]) # verify the arithmetic.

▶ What you'll see: each minority example receives 9 times the weight of a majority example.

In [ ]:
row_weights_b4 = weights_b4[y_b4] # assign each row its class's weight.
print("total class weight:", np.round([row_weights_b4[y_b4 == 0].sum(), row_weights_b4[y_b4 == 1].sum()], 3)) # inspect total force per class.
plt.figure(figsize=(4, 3)) # create a total-weight plot.
plt.bar(["class 0 total", "class 1 total"], [row_weights_b4[y_b4 == 0].sum(), row_weights_b4[y_b4 == 1].sum()], color=["steelblue", "crimson"]) # compare class contributions.
plt.title("Basic 4: balanced total weight") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the two classes now contribute the same total weight.

👀 Takeaway: class weighting fixes imbalance by changing contribution strength, not by changing labels.

### Basic 5 — Compute weighted empirical loss

**Goal.** Apply $L=\frac1m\sum_i w_{y_i}\ell_i$, because weighted ERM is the central formula for imbalanced learning. We build it in 3 steps.

In [ ]:
y_b5 = np.array([0] * 18 + [1] * 2) # define labels.
losses_b5 = np.where(y_b5 == 0, 0.1, 0.9) # set easy majority losses and hard minority losses.
print("first losses:", losses_b5[:5], "last losses:", losses_b5[-2:]) # inspect the per-example losses.

▶ What you'll see: most losses are small, but the two minority losses are large.

In [ ]:
counts_b5 = np.bincount(y_b5) # count classes.
weights_b5 = len(y_b5) / (2 * counts_b5) # compute balanced weights.
plain_loss_b5 = float(np.mean(losses_b5)) # ordinary average loss.
weighted_loss_b5 = float(np.mean(weights_b5[y_b5] * losses_b5)) # weighted average loss.
print("plain loss:", round(plain_loss_b5, 3), "weighted loss:", round(weighted_loss_b5, 3)) # compare objectives.
assert round(plain_loss_b5, 3) == 0.18 # verify the majority-dominated average.
assert round(weighted_loss_b5, 3) == 0.5 # verify the balanced objective.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create a loss comparison figure.
plt.bar(["plain", "weighted"], [plain_loss_b5, weighted_loss_b5], color=["gray", "seagreen"]) # compare objectives.
plt.title("Basic 5: weighted empirical loss") # title the plot.
plt.ylabel("loss") # label the loss axis.
plt.show() # display the chart.

▶ What you'll see: the weighted loss is larger because rare-class errors are no longer diluted.

👀 Takeaway: weighted loss reveals rare-class mistakes that a plain average can hide.

### Basic 6 — Randomly oversample the minority class

**Goal.** Balance classes by repeating minority indices, because oversampling changes the training distribution without editing feature values. We build it in 2 steps.

In [ ]:
rng_b6 = np.random.default_rng(6) # create local randomness for reproducible sampling.
y_b6 = np.array([0] * 18 + [1] * 2) # define imbalanced labels.
idx0_b6 = np.where(y_b6 == 0)[0] # majority indices.
idx1_b6 = np.where(y_b6 == 1)[0] # minority indices.
extra_b6 = rng_b6.choice(idx1_b6, size=len(idx0_b6) - len(idx1_b6), replace=True) # draw repeated minority rows.
print("extra draws:", extra_b6[:6]) # inspect a few repeated minority indices.

▶ What you'll see: oversampling selects the same minority rows many times.

In [ ]:
y_over_b6 = y_b6[np.r_[np.arange(len(y_b6)), extra_b6]] # append repeated minority examples.
counts_over_b6 = np.bincount(y_over_b6) # count the oversampled training labels.
print("oversampled counts:", counts_over_b6) # inspect the balanced result.
assert np.all(counts_over_b6 == [18, 18]) # verify exact balance.
plt.figure(figsize=(4, 3)) # create a count plot.
plt.bar(["class 0", "class 1"], counts_over_b6, color=["steelblue", "crimson"]) # visualize oversampled counts.
plt.title("Basic 6: random oversampling") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the minority class grows to match the majority count.

👀 Takeaway: oversampling balances counts but may repeat the same rare examples many times.

### Basic 7 — Randomly undersample the majority class

**Goal.** Balance classes by keeping fewer majority rows, because undersampling lowers majority dominance without duplicating rare examples. We build it in 2 steps.

In [ ]:
rng_b7 = np.random.default_rng(7) # create reproducible sampling randomness.
y_b7 = np.array([0] * 18 + [1] * 2) # define imbalanced labels.
idx0_b7 = np.where(y_b7 == 0)[0] # majority indices.
idx1_b7 = np.where(y_b7 == 1)[0] # minority indices.
keep0_b7 = rng_b7.choice(idx0_b7, size=len(idx1_b7), replace=False) # keep only as many negatives as positives.
print("kept majority indices:", keep0_b7) # inspect which majority examples survive.

▶ What you'll see: only two majority examples are kept.

In [ ]:
y_under_b7 = y_b7[np.r_[keep0_b7, idx1_b7]] # create the undersampled label vector.
counts_under_b7 = np.bincount(y_under_b7) # count balanced labels.
print("undersampled counts:", counts_under_b7) # inspect the result.
assert np.all(counts_under_b7 == [2, 2]) # verify exact balance.
plt.figure(figsize=(4, 3)) # create a count plot.
plt.bar(["class 0", "class 1"], counts_under_b7, color=["steelblue", "crimson"]) # show balanced small data.
plt.title("Basic 7: random undersampling") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the dataset is balanced but much smaller.

👀 Takeaway: undersampling avoids duplicate positives but throws away majority information.

### Basic 8 — Build one SMOTE point

**Goal.** Interpolate between two minority examples, because SMOTE creates new rare-class feature vectors instead of copying rows. We build it in 2 steps.

In [ ]:
x_i_b8 = np.array([2.0, 1.0]) # choose one minority point.
x_j_b8 = np.array([4.0, 3.0]) # choose a minority neighbor.
lam_b8 = 0.25 # choose an interpolation fraction between 0 and 1.
print("start:", x_i_b8, "neighbor:", x_j_b8, "lambda:", lam_b8) # inspect the ingredients.

▶ What you'll see: the synthetic point will lie one quarter of the way from start to neighbor.

In [ ]:
x_new_b8 = x_i_b8 + lam_b8 * (x_j_b8 - x_i_b8) # apply the SMOTE interpolation formula.
print("synthetic point:", x_new_b8) # inspect the generated feature vector.
assert np.allclose(x_new_b8, [2.5, 1.5]) # verify the hand-checkable interpolation.
plt.figure(figsize=(4, 3)) # create a small geometry plot.
plt.scatter([x_i_b8[0], x_j_b8[0], x_new_b8[0]], [x_i_b8[1], x_j_b8[1], x_new_b8[1]], color=["crimson", "crimson", "orange"], s=[80, 80, 100]) # draw original and synthetic points.
plt.plot([x_i_b8[0], x_j_b8[0]], [x_i_b8[1], x_j_b8[1]], linestyle="--", color="gray") # show the interpolation line.
plt.title("Basic 8: one SMOTE interpolation") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the orange point lies on the line segment between the two minority points.

👀 Takeaway: SMOTE assumes local minority-neighbor line segments remain minority-like.

### Basic 9 — Compute precision and recall

**Goal.** Count TP, FP, and FN for a thresholded classifier, because rare-class evaluation needs more than accuracy. We build it in 2 steps.

In [ ]:
y_b9 = np.array([0, 0, 1, 0, 1, 1]) # define validation labels.
scores_b9 = np.array([0.10, 0.30, 0.35, 0.60, 0.70, 0.90]) # define predicted scores.
pred_b9 = scores_b9 >= 0.50 # threshold scores into positive decisions.
print("predictions:", pred_b9.astype(int)) # inspect final decisions.

▶ What you'll see: threshold 0.50 predicts three positives.

In [ ]:
tp_b9 = np.sum(pred_b9 & (y_b9 == 1)) # true positives.
fp_b9 = np.sum(pred_b9 & (y_b9 == 0)) # false positives.
fn_b9 = np.sum((~pred_b9) & (y_b9 == 1)) # false negatives.
precision_b9 = tp_b9 / (tp_b9 + fp_b9) # positive predictive value.
recall_b9 = tp_b9 / (tp_b9 + fn_b9) # rare-class recovery rate.
print("TP FP FN:", tp_b9, fp_b9, fn_b9) # inspect confusion pieces.
print("precision:", round(float(precision_b9), 3), "recall:", round(float(recall_b9), 3)) # inspect metrics.
assert round(float(precision_b9), 3) == 0.667 # verify 2/(2+1).
assert round(float(recall_b9), 3) == 0.667 # verify 2/(2+1).

▶ What you'll see: one false positive and one false negative create equal precision and recall.

In [ ]:
metric_names_b9 = ["precision", "recall"] # name the rare-class metrics.
metric_values_b9 = [precision_b9, recall_b9] # reuse the computed scores.
plt.figure(figsize=(4, 3)) # create a compact metric comparison plot.
plt.bar(metric_names_b9, metric_values_b9, color=["steelblue", "darkorange"]) # compare false alarms and missed positives.
plt.ylim(0, 1) # keep the metric scale familiar.
plt.ylabel("score") # label the score axis.
plt.title("Basic 9: precision and recall") # title the plot.
plt.show() # display the chart.

▶ What you'll see: precision and recall have matching bar heights because both are 2/3.

👀 Takeaway: precision measures false alarms; recall measures missed rare positives.

### Basic 10 — Sweep a decision threshold

**Goal.** Try multiple thresholds, because an imbalanced model's probability scores need a decision rule matched to the cost of misses and false alarms. We build it in 3 steps.

In [ ]:
y_b10 = np.array([0, 0, 1, 0, 1, 1]) # validation labels.
scores_b10 = np.array([0.10, 0.30, 0.35, 0.60, 0.70, 0.90]) # predicted scores.
thresholds_b10 = np.array([0.30, 0.50, 0.70]) # candidate thresholds.
print("thresholds:", thresholds_b10) # inspect threshold grid.

▶ What you'll see: three possible cutoffs from permissive to strict.

In [ ]:
recalls_b10 = [] # store rare-class recall for each threshold.
precisions_b10 = [] # store precision for each threshold.
for th_b10 in thresholds_b10: # evaluate each cutoff.
    pred_b10 = scores_b10 >= th_b10 # threshold scores.
    tp_b10 = np.sum(pred_b10 & (y_b10 == 1)) # count true positives.
    fp_b10 = np.sum(pred_b10 & (y_b10 == 0)) # count false positives.
    fn_b10 = np.sum((~pred_b10) & (y_b10 == 1)) # count false negatives.
    precisions_b10.append(tp_b10 / (tp_b10 + fp_b10) if tp_b10 + fp_b10 else 0.0) # compute precision.
    recalls_b10.append(tp_b10 / (tp_b10 + fn_b10) if tp_b10 + fn_b10 else 0.0) # compute recall.
print("precision:", np.round(precisions_b10, 3)) # inspect false-alarm tradeoff.
print("recall:", np.round(recalls_b10, 3)) # inspect missed-positive tradeoff.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create a threshold sweep plot.
plt.plot(thresholds_b10, precisions_b10, marker="o", label="precision") # plot precision by threshold.
plt.plot(thresholds_b10, recalls_b10, marker="s", label="recall") # plot recall by threshold.
plt.ylim(0, 1.05) # keep metric scale fixed.
plt.title("Basic 10: threshold sweep") # title the plot.
plt.xlabel("threshold") # label the cutoff axis.
plt.legend() # show metric labels.
plt.show() # display the chart.

▶ What you'll see: stricter thresholds usually reduce recall and may improve precision.

👀 Takeaway: threshold selection is part of imbalanced classification, not an afterthought.

## 🟡 Easy

### Easy 1 — Compare weighting to oversampling

**Goal.** Show that balanced class weights and exact oversampling can give the same average loss when duplicated rows carry identical losses. We build it in 3 steps.

In [ ]:
y_e1 = np.array([0] * 6 + [1] * 2) # create a 6:2 imbalance.
losses_e1 = np.array([0.2, 0.1, 0.3, 0.2, 0.1, 0.2, 0.8, 0.6]) # define per-example losses.
counts_e1 = np.bincount(y_e1) # count classes.
weights_e1 = len(y_e1) / (2 * counts_e1) # balanced weights.
print("weights:", np.round(weights_e1, 3)) # inspect class multipliers.

▶ What you'll see: the minority class gets three times the majority weight.

In [ ]:
weighted_e1 = float(np.mean(weights_e1[y_e1] * losses_e1)) # weighted ERM objective.
idx_over_e1 = np.r_[np.arange(len(y_e1)), [6, 7, 6, 7]] # repeat minority rows until counts are 6 and 6.
over_loss_e1 = float(np.mean(losses_e1[idx_over_e1])) # ordinary average on oversampled data.
print("weighted loss:", round(weighted_e1, 3), "oversampled loss:", round(over_loss_e1, 3)) # compare objectives.
assert round(weighted_e1, 3) == round(over_loss_e1, 3) == 0.442 # verify equivalence in this exact setup.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create an objective comparison chart.
plt.bar(["weighted", "oversampled"], [weighted_e1, over_loss_e1], color=["seagreen", "orange"]) # compare the two loss estimates.
plt.title("Easy 1: equivalent objective in a toy case") # title the plot.
plt.ylabel("average loss") # label the loss axis.
plt.show() # display the chart.

▶ What you'll see: both bars match because oversampling repeated the same minority losses exactly.

👀 Takeaway: weighting and oversampling can express similar objectives, but oversampling changes data exposure while weighting changes loss multipliers.

### Easy 2 — Choose a threshold by F1

**Goal.** Select the validation threshold with the best F1 score, because imbalanced tasks often need a precision-recall compromise. We build it in 3 steps.

In [ ]:
y_e2 = np.array([0, 0, 0, 1, 0, 1, 0, 1, 1, 1]) # validation labels.
scores_e2 = np.array([0.05, 0.12, 0.18, 0.22, 0.31, 0.40, 0.55, 0.63, 0.72, 0.90]) # validation scores.
thresholds_e2 = np.array([0.2, 0.4, 0.6, 0.8]) # candidate cutoffs.
print("positive count:", int(y_e2.sum())) # inspect rare-class support.

▶ What you'll see: the validation set contains five positives.

In [ ]:
f1s_e2 = [] # store F1 scores.
for th_e2 in thresholds_e2: # loop over thresholds.
    pred_e2 = scores_e2 >= th_e2 # make decisions.
    tp_e2 = np.sum(pred_e2 & (y_e2 == 1)) # true positives.
    fp_e2 = np.sum(pred_e2 & (y_e2 == 0)) # false positives.
    fn_e2 = np.sum((~pred_e2) & (y_e2 == 1)) # false negatives.
    prec_e2 = tp_e2 / (tp_e2 + fp_e2) if tp_e2 + fp_e2 else 0.0 # precision.
    rec_e2 = tp_e2 / (tp_e2 + fn_e2) if tp_e2 + fn_e2 else 0.0 # recall.
    f1s_e2.append(2 * prec_e2 * rec_e2 / (prec_e2 + rec_e2) if prec_e2 + rec_e2 else 0.0) # harmonic mean.
print("F1 scores:", np.round(f1s_e2, 3)) # inspect validation scores.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
best_idx_e2 = int(np.argmax(f1s_e2)) # find the best threshold index.
best_threshold_e2 = float(thresholds_e2[best_idx_e2]) # read the best threshold.
print("best threshold:", best_threshold_e2) # inspect the selected cutoff.
assert best_threshold_e2 == 0.2 # verify the best cutoff for this toy validation set.
plt.figure(figsize=(4, 3)) # create an F1 sweep plot.
plt.plot(thresholds_e2, f1s_e2, marker="o", color="purple") # plot F1 by threshold.
plt.axvline(best_threshold_e2, color="red", linestyle="--") # mark the selected threshold.
plt.title("Easy 2: choose threshold by F1") # title the plot.
plt.xlabel("threshold") # label the cutoff axis.
plt.ylabel("F1") # label the metric axis.
plt.show() # display the chart.

▶ What you'll see: the lowest tested threshold gives the highest F1 in this validation set.

👀 Takeaway: imbalanced classification often tunes the decision threshold separately from training.

### Easy 3 — Build a small SMOTE cloud

**Goal.** Generate several synthetic minority points, because SMOTE's geometric assumption is easiest to inspect in two dimensions. We build it in 3 steps.

In [ ]:
X_min_e3 = np.array([[1.0, 1.0], [1.4, 1.2], [1.2, 1.7], [1.8, 1.6]]) # four minority examples.
rng_e3 = np.random.default_rng(3) # local generator for reproducible interpolation.
print("minority shape:", X_min_e3.shape) # inspect starting data.

▶ What you'll see: only four original minority points are available.

In [ ]:
X_syn_e3 = [] # store synthetic examples.
for t_e3 in range(12): # create twelve synthetic points.
    i_e3 = rng_e3.integers(0, len(X_min_e3)) # choose a base minority point.
    j_e3 = (i_e3 + rng_e3.integers(1, len(X_min_e3))) % len(X_min_e3) # choose a different neighbor index.
    lam_e3 = rng_e3.random() # draw interpolation fraction.
    X_syn_e3.append(X_min_e3[i_e3] + lam_e3 * (X_min_e3[j_e3] - X_min_e3[i_e3])) # append SMOTE point.
X_syn_e3 = np.array(X_syn_e3) # convert list to array.
print("synthetic shape:", X_syn_e3.shape) # inspect generated data size.
assert X_syn_e3.shape == (12, 2) # verify expected number of points.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create a two-dimensional point plot.
plt.scatter(X_min_e3[:, 0], X_min_e3[:, 1], s=90, color="crimson", label="original") # plot original minority points.
plt.scatter(X_syn_e3[:, 0], X_syn_e3[:, 1], s=35, color="orange", label="synthetic") # plot SMOTE points.
plt.title("Easy 3: SMOTE cloud") # title the figure.
plt.xlabel("x1") # label feature 1.
plt.ylabel("x2") # label feature 2.
plt.legend() # show point types.
plt.show() # display the chart.

▶ What you'll see: synthetic points fill gaps among the original minority examples.

👀 Takeaway: SMOTE adds local minority support without creating exact duplicates.

### Easy 4 — Compare overfitting risk from duplicates

**Goal.** Count how many times each minority row appears after oversampling, because duplicate exposure can make a learner memorize rare examples. We build it in 3 steps.

In [ ]:
rng_e4 = np.random.default_rng(4) # reproducible oversampling.
y_e4 = np.array([0] * 12 + [1] * 3) # create a 12:3 imbalance.
idx1_e4 = np.where(y_e4 == 1)[0] # minority row indices.
extra_e4 = rng_e4.choice(idx1_e4, size=9, replace=True) # add nine duplicates to balance class counts.
print("extra minority indices:", extra_e4) # inspect repeated rows.

▶ What you'll see: the same three minority rows are reused many times.

In [ ]:
all_min_indices_e4 = np.r_[idx1_e4, extra_e4] # include original and duplicated minority exposures.
unique_e4, counts_e4 = np.unique(all_min_indices_e4, return_counts=True) # count exposure per minority row.
print("minority exposure counts:", dict(zip(unique_e4, counts_e4))) # inspect memorization pressure.
assert counts_e4.sum() == 12 # verify minority class now has twelve exposures.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create an exposure-count plot.
plt.bar([str(i) for i in unique_e4], counts_e4, color="orange") # show how often each minority row is seen.
plt.title("Easy 4: duplicate exposure after oversampling") # title the plot.
plt.xlabel("minority row index") # label x-axis.
plt.ylabel("training exposures") # label exposure count.
plt.show() # display the chart.

▶ What you'll see: oversampling balances classes by showing some exact minority rows repeatedly.

👀 Takeaway: oversampling helps class balance but can amplify noise in individual rare examples.

### Easy 5 — Use a cost-sensitive score

**Goal.** Compare two thresholds using a false-negative cost larger than false-positive cost, because many imbalanced tasks care more about missing rare positives. We build it in 3 steps.

In [ ]:
y_e5 = np.array([0, 0, 0, 1, 0, 1, 0, 1]) # validation labels.
scores_e5 = np.array([0.05, 0.20, 0.30, 0.35, 0.45, 0.52, 0.60, 0.80]) # validation scores.
thresholds_e5 = np.array([0.3, 0.5]) # compare permissive and stricter thresholds.
fn_cost_e5 = 5.0 # make missed positives expensive.
fp_cost_e5 = 1.0 # make false alarms cheaper.
print("costs FN/FP:", fn_cost_e5, fp_cost_e5) # inspect decision costs.

▶ What you'll see: a false negative costs five times as much as a false positive.

In [ ]:
costs_e5 = [] # store total cost per threshold.
for th_e5 in thresholds_e5: # evaluate each decision rule.
    pred_e5 = scores_e5 >= th_e5 # threshold scores.
    fp_e5 = np.sum(pred_e5 & (y_e5 == 0)) # count false alarms.
    fn_e5 = np.sum((~pred_e5) & (y_e5 == 1)) # count misses.
    costs_e5.append(fp_cost_e5 * fp_e5 + fn_cost_e5 * fn_e5) # compute total operational cost.
print("costs:", costs_e5) # inspect the cost comparison.
assert costs_e5 == [3.0, 6.0] # verify the permissive threshold is cheaper here.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
best_threshold_e5 = float(thresholds_e5[int(np.argmin(costs_e5))]) # choose lowest-cost threshold.
print("best threshold by cost:", best_threshold_e5) # inspect the decision.
plt.figure(figsize=(4, 3)) # create a cost comparison plot.
plt.bar([str(t) for t in thresholds_e5], costs_e5, color=["seagreen", "gray"]) # plot costs.
plt.title("Easy 5: cost-sensitive threshold") # title the plot.
plt.xlabel("threshold") # label threshold axis.
plt.ylabel("total cost") # label cost axis.
plt.show() # display the chart.

▶ What you'll see: the lower threshold wins because avoiding missed positives matters more.

👀 Takeaway: imbalanced classification should optimize the decision cost, not only a generic score.

## 🔴 Advanced

### Advanced 1 — Train weighted logistic regression from scratch

**Goal.** Fit a one-feature logistic model with weighted gradients, because class weights enter learning by multiplying each example's loss derivative. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(1) # reproducible synthetic data.
X0_a1 = rng_a1.normal(loc=-0.5, scale=0.7, size=60) # majority feature values.
X1_a1 = rng_a1.normal(loc=1.2, scale=0.5, size=6) # minority feature values.
x_a1 = np.r_[X0_a1, X1_a1] # combine features.
y_a1 = np.r_[np.zeros(len(X0_a1)), np.ones(len(X1_a1))] # combine labels.
print("counts:", np.bincount(y_a1.astype(int))) # inspect imbalance.

▶ What you'll see: the positive class has one tenth as many examples as the negative class.

In [ ]:
counts_a1 = np.bincount(y_a1.astype(int)) # count classes.
w_class_a1 = len(y_a1) / (2 * counts_a1) # balanced class weights.
w_row_a1 = w_class_a1[y_a1.astype(int)] # row weights.
beta_a1 = np.array([0.0, 0.0]) # intercept and slope.
print("class weights:", np.round(w_class_a1, 3)) # inspect multipliers.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
for step_a1 in range(600): # run batch gradient descent.
    z_a1 = beta_a1[0] + beta_a1[1] * x_a1 # linear score.
    p_a1 = 1 / (1 + np.exp(-z_a1)) # sigmoid probability.
    grad0_a1 = np.mean(w_row_a1 * (p_a1 - y_a1)) # weighted intercept gradient.
    grad1_a1 = np.mean(w_row_a1 * (p_a1 - y_a1) * x_a1) # weighted slope gradient.
    beta_a1 -= 0.2 * np.array([grad0_a1, grad1_a1]) # gradient descent update.
print("beta:", np.round(beta_a1, 3)) # inspect learned parameters.
assert beta_a1[1] > 0 # verify higher x means more positive probability.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
grid_a1 = np.linspace(-2.5, 2.5, 100) # feature grid for plotting probabilities.
prob_a1 = 1 / (1 + np.exp(-(beta_a1[0] + beta_a1[1] * grid_a1))) # fitted probability curve.
plt.figure(figsize=(5, 3)) # create a model plot.
plt.scatter(x_a1[y_a1 == 0], y_a1[y_a1 == 0], alpha=0.5, color="steelblue", label="class 0") # plot negatives.
plt.scatter(x_a1[y_a1 == 1], y_a1[y_a1 == 1], alpha=0.8, color="crimson", label="class 1") # plot positives.
plt.plot(grid_a1, prob_a1, color="black", label="weighted logistic") # draw fitted probability curve.
plt.title("Advanced 1: weighted logistic regression") # title the plot.
plt.xlabel("feature") # label feature axis.
plt.ylabel("P(class 1)") # label probability axis.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: the fitted probability curve rises through the minority region despite few positive examples.

👀 Takeaway: class weights multiply gradients, so rare examples can pull parameters with comparable total force.

### Advanced 2 — Compare weighted and unweighted training losses

**Goal.** Train two logistic models from the same data and compare minority recall, because a lower unweighted objective may still under-serve positives. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(2) # reproducible data.
x_a2 = np.r_[rng_a2.normal(-0.3, 0.8, 80), rng_a2.normal(0.9, 0.6, 8)] # overlapping features.
y_a2 = np.r_[np.zeros(80), np.ones(8)] # imbalanced labels.
weights_bal_a2 = len(y_a2) / (2 * np.bincount(y_a2.astype(int))) # balanced class weights.
print("balanced weights:", np.round(weights_bal_a2, 3)) # inspect multipliers.

▶ What you'll see: positive examples receive much larger loss multipliers.

In [ ]:
def train_logreg_a2(row_weights_a2):
    beta_a2 = np.array([0.0, 0.0]) # initialize intercept and slope.
    for step_a2 in range(500): # batch gradient descent.
        p_a2 = 1 / (1 + np.exp(-(beta_a2[0] + beta_a2[1] * x_a2))) # probabilities.
        beta_a2 -= 0.15 * np.array([np.mean(row_weights_a2 * (p_a2 - y_a2)), np.mean(row_weights_a2 * (p_a2 - y_a2) * x_a2)]) # weighted gradient step.
    return beta_a2 # return trained parameters.
beta_plain_a2 = train_logreg_a2(np.ones_like(y_a2)) # train unweighted model.
beta_weighted_a2 = train_logreg_a2(weights_bal_a2[y_a2.astype(int)]) # train weighted model.
print("plain beta:", np.round(beta_plain_a2, 3), "weighted beta:", np.round(beta_weighted_a2, 3)) # compare parameters.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
prob_plain_a2 = 1 / (1 + np.exp(-(beta_plain_a2[0] + beta_plain_a2[1] * x_a2))) # plain probabilities.
prob_weighted_a2 = 1 / (1 + np.exp(-(beta_weighted_a2[0] + beta_weighted_a2[1] * x_a2))) # weighted probabilities.
recall_plain_a2 = np.mean((prob_plain_a2[y_a2 == 1] >= 0.5)) # minority recall at 0.5.
recall_weighted_a2 = np.mean((prob_weighted_a2[y_a2 == 1] >= 0.5)) # weighted-model minority recall.
print("minority recall plain/weighted:", round(float(recall_plain_a2), 3), round(float(recall_weighted_a2), 3)) # inspect rare-class recovery.
assert recall_weighted_a2 >= recall_plain_a2 # weighted training should not recover fewer positives in this toy setup.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(4, 3)) # create a recall comparison chart.
plt.bar(["plain", "weighted"], [recall_plain_a2, recall_weighted_a2], color=["gray", "seagreen"]) # compare minority recall.
plt.ylim(0, 1.05) # use metric scale.
plt.title("Advanced 2: weighting and minority recall") # title the plot.
plt.ylabel("recall at threshold 0.5") # label metric axis.
plt.show() # display the chart.

▶ What you'll see: the weighted model recovers more minority positives at the same threshold.

👀 Takeaway: the right training objective is the one whose validation behavior matches the rare-class goal.

### Advanced 3 — Detect SMOTE boundary risk

**Goal.** Show that interpolation can cross into majority territory, because SMOTE is local and can be unsafe when minority points surround the wrong region. We build it in 4 steps.

In [ ]:
X_major_a3 = np.array([[0.0, 0.0], [0.2, 0.1], [-0.2, 0.0], [0.1, -0.2]]) # majority cluster near origin.
minor_a_a3 = np.array([-1.0, 0.0]) # one minority point.
minor_b_a3 = np.array([1.0, 0.0]) # another minority point across the majority cluster.
lam_grid_a3 = np.linspace(0, 1, 11) # interpolation fractions.
path_a3 = minor_a_a3 + lam_grid_a3[:, None] * (minor_b_a3 - minor_a_a3) # synthetic path.
print("middle synthetic point:", path_a3[5]) # inspect the point halfway between minority examples.

▶ What you'll see: the midpoint is at the origin, exactly where the majority cluster lives.

In [ ]:
dist_to_major_a3 = np.min(np.linalg.norm(path_a3[:, None, :] - X_major_a3[None, :, :], axis=2), axis=1) # nearest majority distance for each synthetic point.
risky_a3 = dist_to_major_a3 < 0.25 # mark synthetic points too close to majority examples.
print("risky synthetic count:", int(risky_a3.sum())) # inspect boundary danger.
assert int(risky_a3.sum()) >= 2 # verify some interpolations are risky.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # create a geometry plot.
plt.scatter(X_major_a3[:, 0], X_major_a3[:, 1], s=80, color="steelblue", label="majority") # plot majority cluster.
plt.scatter([minor_a_a3[0], minor_b_a3[0]], [minor_a_a3[1], minor_b_a3[1]], s=100, color="crimson", label="minority endpoints") # plot minority endpoints.
plt.scatter(path_a3[:, 0], path_a3[:, 1], c=np.where(risky_a3, "red", "orange"), s=35, label="SMOTE path") # color risky synthetics.
plt.title("Advanced 3: SMOTE can cross majority territory") # title the plot.
plt.legend() # show labels.
plt.show() # display chart.

▶ What you'll see: red synthetic points appear near the majority cluster between two distant minority endpoints.

In [ ]:
safe_fraction_a3 = 1 - risky_a3.mean() # compute share of interpolations away from majority cluster.
print("safe fraction:", round(float(safe_fraction_a3), 3)) # inspect how much of the path is safe.

▶ What you'll see: not every point on a minority-minority segment is necessarily a safe minority example.

👀 Takeaway: SMOTE works best with genuinely local minority neighbors, not distant points separated by majority density.

### Advanced 4 — Tune oversampling amount on validation data

**Goal.** Sweep minority oversampling ratios and evaluate validation F1, because more balancing is not automatically better. We build it in 4 steps.

In [ ]:
base_losses_pos_a4 = np.array([0.9, 0.7, 0.6, 0.5]) # toy positive validation losses after training variants.
ratios_a4 = np.array([0.25, 0.5, 1.0, 2.0]) # minority-to-majority exposure ratios to compare.
precision_a4 = np.array([0.90, 0.82, 0.70, 0.55]) # plausible precision as oversampling increases.
recall_a4 = np.array([0.30, 0.55, 0.80, 0.90]) # plausible recall as oversampling increases.
print("ratios:", ratios_a4) # inspect sweep grid.

▶ What you'll see: the grid ranges from under-balanced to minority-heavy training exposure.

In [ ]:
f1_a4 = 2 * precision_a4 * recall_a4 / (precision_a4 + recall_a4) # harmonic mean of precision and recall.
best_idx_a4 = int(np.argmax(f1_a4)) # find validation winner.
print("F1 by ratio:", np.round(f1_a4, 3)) # inspect validation tradeoff.
print("best ratio:", ratios_a4[best_idx_a4]) # inspect selected oversampling amount.
assert ratios_a4[best_idx_a4] == 1.0 # verify the balanced ratio wins this toy sweep.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # create a validation sweep plot.
plt.plot(ratios_a4, precision_a4, marker="o", label="precision") # plot false-alarm behavior.
plt.plot(ratios_a4, recall_a4, marker="s", label="recall") # plot rare-class recovery.
plt.plot(ratios_a4, f1_a4, marker="^", label="F1") # plot combined validation score.
plt.axvline(ratios_a4[best_idx_a4], color="red", linestyle="--", label="best") # mark selected ratio.
plt.title("Advanced 4: oversampling ratio sweep") # title the plot.
plt.xlabel("minority exposure / majority exposure") # label ratio axis.
plt.ylim(0, 1.05) # keep metric scale fixed.
plt.legend() # show metric labels.
plt.show() # display the chart.

▶ What you'll see: recall rises with more minority exposure, but precision falls, creating a best middle point.

In [ ]:
selected_score_a4 = float(f1_a4[best_idx_a4]) # store selected validation score.
print("selected validation F1:", round(selected_score_a4, 3)) # inspect chosen score.

▶ What you'll see: the selected ratio is justified by validation, not by the prettiest training balance.

👀 Takeaway: resampling strength is a hyperparameter and should be selected on held-out data.

### Advanced 5 — Compare full decision scores with cost

**Goal.** Combine weighted loss, operational cost, validation F1, and a stability adjustment, because the lesson's decision logic ranks the full score rather than a raw training fragment. We build it in 4 steps.

In [ ]:
losses_a5 = np.array([0.202, 0.083, 0.488]) # verified toy per-example losses from the lesson block.
raw_risk_a5 = float(np.mean(losses_a5)) # compute empirical risk.
cost_a5 = 0.100 # method cost from the lesson block.
score_a5 = round(raw_risk_a5, 3) + cost_a5 # full baseline score using the displayed rounded risk.
print("raw risk:", round(raw_risk_a5, 3), "score:", round(score_a5, 3)) # inspect components.
assert round(raw_risk_a5, 3) == 0.258 # verify lesson arithmetic.
assert round(score_a5, 3) == 0.358 # verify score with cost.

▶ What you'll see: the raw average is only part of the selection score.

In [ ]:
alternative_a5 = 0.402 # flexible alternative score from the lesson block.
gap_a5 = alternative_a5 - score_a5 # absolute gap.
relative_gap_a5 = gap_a5 / alternative_a5 # relative evidence scale.
print("gap:", round(gap_a5, 3), "relative gap:", round(relative_gap_a5, 3)) # inspect comparison strength.
assert round(gap_a5, 3) == 0.044 # verify the lesson gap.
assert round(relative_gap_a5, 3) == 0.109 # verify relative gap.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
stable_a5 = 0.80 * score_a5 # stabilizing knob reduces decision score by 20%.
choices_a5 = np.array([score_a5, alternative_a5, stable_a5]) # collect candidate scores.
labels_a5 = ["baseline", "flexible", "stable"] # names for candidates.
winner_a5 = labels_a5[int(np.argmin(choices_a5))] # lower score wins.
print("stable score:", round(stable_a5, 3), "winner:", winner_a5) # inspect final decision.
assert round(stable_a5, 3) == 0.286 # verify stable score.
assert winner_a5 == "stable" # verify the full-score winner.

▶ What you'll see: the printed values or plot expose the intermediate quantity used by the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # create a final score comparison plot.
plt.bar(labels_a5, choices_a5, color=["gray", "orange", "seagreen"]) # compare candidate scores.
plt.title("Advanced 5: choose by full decision score") # title the plot.
plt.ylabel("lower is better") # label score scale.
plt.show() # display the chart.

▶ What you'll see: the stable option wins only after raw loss, cost, and comparison are kept on the same scale.

👀 Takeaway: imbalance fixes should be selected by their full validation-aware decision score, not by raw fit alone.